## Cell 1: reload raw data (same as before)

In [1]:
import scipy.io as sio
import numpy as np
import pandas as pd

BATTERIES = ['B0005', 'B0006', 'B0007', 'B0018']

def find_path(fname):
    return f'C:/Users/santh/battery-rul-project/data/raw/{fname}.mat'

def load_battery(fname):
    m = sio.loadmat(find_path(fname), struct_as_record=False, squeeze_me=True)
    return m[fname].cycle

raw = {b: load_battery(b) for b in BATTERIES}

## Cell 2: Fix #1: drop the malformed last cycle (idx 615) for B0005/6/7

In [2]:
# B0005, B0006, B0007 lo last cycle (idx 615) truncated/malformed ga undi — drop
cleaned_raw = {}
for b in BATTERIES:
    cycles = list(raw[b])
    if b in ['B0005', 'B0006', 'B0007']:
        cycles = cycles[:-1]   # last cycle (idx 615) remove
    cleaned_raw[b] = cycles

print({b: len(cleaned_raw[b]) for b in cleaned_raw})

{'B0005': 615, 'B0006': 615, 'B0007': 615, 'B0018': 319}


## Cell 3: Fix #2: drop the voltage-spike outlier cycle (idx 84, charge) for B0005/6/7

In [3]:
# cycle_idx 84 (charge) lo B0005/6/7 anni lo voltage 8V+ spike undi — drop that cycle
for b in ['B0005', 'B0006', 'B0007']:
    cleaned_raw[b] = [c for i, c in enumerate(cleaned_raw[b]) if i != 84]

print({b: len(cleaned_raw[b]) for b in cleaned_raw})

{'B0005': 614, 'B0006': 614, 'B0007': 614, 'B0018': 319}


## Cell 4: Fix #3: fix the 2 NaN points in B0018 cycle_idx 114

In [4]:
# B0018, cycle_idx 114 (charge) lo 2 NaN points unnayi — linear interpolation tho fill
for i, c in enumerate(cleaned_raw['B0018']):
    if c.type == 'charge' and i == 114:
        d = c.data
        for field in ['Voltage_measured', 'Current_measured', 'Temperature_measured']:
            arr = np.array(getattr(d, field), dtype=float)
            nan_mask = np.isnan(arr)
            if nan_mask.any():
                arr[nan_mask] = np.interp(np.flatnonzero(nan_mask), np.flatnonzero(~nan_mask), arr[~nan_mask])
                setattr(d, field, arr)
        print('Fixed NaNs in B0018 cycle 114')

Fixed NaNs in B0018 cycle 114


## Cell 5: Fix #4: resample every charge/discharge cycle to a fixed length (padding/interpolation problem solve)

In [5]:
def resample_series(arr, n_points=300):
    arr = np.array(arr, dtype=float).flatten()
    if len(arr) < 2:
        return np.full(n_points, np.nan)
    x_old = np.linspace(0, 1, len(arr))
    x_new = np.linspace(0, 1, n_points)
    return np.interp(x_new, x_old, arr)

rows = []
for b in BATTERIES:
    for i, c in enumerate(cleaned_raw[b]):
        if c.type not in ('charge', 'discharge'):
            continue
        d = c.data
        row = {
            'battery': b, 'cycle_idx': i, 'type': c.type,
            'voltage': resample_series(d.Voltage_measured),
            'current': resample_series(d.Current_measured),
            'temperature': resample_series(d.Temperature_measured),
        }
        if c.type == 'discharge':
            cap = np.array(d.Capacity).flatten()
            row['capacity_Ah'] = float(cap[0]) if cap.size else np.nan
        rows.append(row)

clean_df = pd.DataFrame(rows)
clean_df.head()

,battery,cycle_idx,type,voltage,current,temperature,capacity_Ah
0,B0005,0,charge,"[3.873017221300996, 4.008090834326928, 4.02682...","[-0.001200660698297908, 1.5104002145198037, 1....","[24.65535783391511, 24.687131655749603, 24.721...",NaN
1,B0005,1,discharge,"[4.191491807505295, 4.191004927950373, 4.12360...","[-0.004901589207462691, -0.002657367145453474,...","[24.330033885570543, 24.327385288702907, 24.34...",1.856487
2,B0005,2,charge,"[3.3250546568448542, 3.456813992633407, 3.4947...","[0.00030204673114322896, 1.5098550262131725, 1...","[29.341850509195503, 29.34047586246449, 29.297...",NaN
3,B0005,3,discharge,"[4.189773213846608, 4.1891915832591025, 4.1250...","[2.125117981080765e-05, -0.0005661765024619325...","[24.697751935729325, 24.69005382346846, 24.701...",1.846327
4,B0005,4,charge,"[3.3526036599987754, 3.4801018744308387, 3.515...","[0.0019896423395958083, 1.5114793033939828, 1....","[29.553300732538816, 29.543702430255458, 29.50...",NaN


## Cell 6: save the cleaned dataset

In [6]:
clean_df.to_pickle('C:/Users/santh/battery-rul-project/data/processed/clean_battery_data.pkl')
print('Saved cleaned dataset.')

Saved cleaned dataset.
